In [ ]:
# ── Colab Setup ───────────────────────────────────────────────────────────────
# Run this cell first every time you open a new Colab session.
# Clones the repo (data and artifacts included) and configures the environment.
import os, sys, importlib

REPO_URL  = "https://github.com/tackes/Modern-Time-Series-Forecasting-Cohort.git"
REPO_PATH = "/content/packt-modern-time-series"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone -q {REPO_URL} {REPO_PATH}")

# Stay in instructor_notebooks so Path().resolve().parent resolves to repo root
os.chdir(f"{REPO_PATH}/instructor_notebooks")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Colab already has GPU torch — only install CPU build if torch is missing
if importlib.util.find_spec("torch") is None:
    os.system("pip install -q torch --index-url https://download.pytorch.org/whl/cpu")
os.system("pip install -q neuralforecast")

print(f"✓ Setup complete — {os.getcwd()}")

# Module 6 — Deep Learning with NHITS
**Type:** [Code With Me]  
**Time:** 40 minutes  
**Job:** Implement a global neural forecasting model (NHITS). Compare it against the LightGBM and baseline floors. Be honest about what deep learning adds — and what it costs.

---

> **Instructor note:** NHITS is the highest-risk runtime cell in the workshop. Structure the module defensively:
> - 6.1–6.4: Setup, framing, load params (5 min)
> - 6.5–6.7: Configure NHITS, run CV — **have the Red Path ready before you press Run** (15 min)
> - 6.8–6.10: Reshape, plot, score (10 min, Code With Me)
> - 6.11–6.13: Load full artifact, leaderboard, enterprise cliff (10 min)
>
> If NHITS training exceeds 90 seconds, interrupt immediately and say: "This is why the Red Path exists." Then load the checkpoint. Do not wait for the cell to finish.

---
## 6.1 — Setup
**[Watch Only]**

> **Instructor note (1 min):** Run silently. New imports: `NeuralForecast` and `NHITS`. Note that `pytorch-lightning` is the training backend — it is installed via `requirements.txt` but not imported explicitly.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams['figure.figsize'] = (14, 4)
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.utils import PredictionIntervals as NFPredictionIntervals

from config import (
    ARTIFACT_DIR,
    PARAMS_DIR,
    HORIZON,
    N_WINDOWS,
    STEP_SIZE,
    REFIT,
    MICRO_SUBSET_N,
    WORKSHOP_SUBSET_N,
    RANDOM_SEED,
    USE_TUNED_PARAMS,
    USE_INTERVALS,
)
from src.checkpointing import load_checkpoint
from src.evaluation import score_forecasts, build_leaderboard
from src.schemas import validate_forecast
from src.workshop_utils import get_micro_subset

print("Setup complete.")
print(f"USE_INTERVALS = {USE_INTERVALS}")

---
## 6.2 — Load Panel and Micro Subset
**[Watch Only]**

> **Instructor note (30 sec):** Same pattern. Load from validated artifact, select micro subset.

In [ ]:
panel = load_checkpoint("03_validated_panel")
micro, top_series = get_micro_subset(panel, n=MICRO_SUBSET_N)

print(f"Micro panel: {micro['unique_id'].nunique()} series, {len(micro):,} rows")

---
## 6.3 — Why NHITS?
**[Watch Only]**

> **Instructor note (2 min):** Frame the model choice before writing any code. The question is not "is NHITS better than LightGBM?" — it is "what problem does a neural model solve that tree-based models do not?"

NHITS (Neural Hierarchical Interpolation for Time Series) is a global neural model designed specifically for multi-step ahead forecasting. Published at AAAI 2023.

**What it does differently from LightGBM:**
- Learns across multiple temporal scales simultaneously using hierarchical pooling
- Produces the full 28-step horizon in a single forward pass — no recursive error accumulation
- Captures non-linear interactions that tree-based models with fixed lag windows miss

**What it costs:**
- Training time is not parallelizable in the same way as LightGBM — it is a neural network with gradient updates
- Hyperparameter sensitivity is higher — `max_steps`, `input_size`, and learning rate all affect stability
- Interpretability is lower — you cannot extract feature importances the way you can from a tree model

**The production question:**  
NHITS is worth the cost when your demand patterns have complex multi-scale dependencies that lag features cannot capture — long promotional build-ups, multi-week seasonality stacking, or cross-series structural patterns. For stable weekly seasonal demand, LightGBM often wins on the ROI calculation.

---
## 6.4 — Load NHITS Parameters
**[Watch Only]**

> **Instructor note (1 min):** Load and show the params. Point at `max_steps = 300` explicitly — this is the primary runtime lever. If the training cell times out, this is the first number to cut. Do not change it live unless necessary.

In [ ]:
params_path = PARAMS_DIR / "nhits_tuned.json"
with open(params_path) as f:
    nhits_params = json.load(f)

# Strip comment keys before passing to model constructor
nhits_params = {k: v for k, v in nhits_params.items() if not k.startswith("_")}

# Lock random seed from config
nhits_params["random_seed"] = RANDOM_SEED

# h must match HORIZON from config — override params file value
nhits_params["h"] = HORIZON

print("NHITS parameters:")
for k, v in nhits_params.items():
    print(f"  {k:<30}: {v}")

print(f"\n  Runtime lever: max_steps = {nhits_params['max_steps']}")
print(f"  Reduce to 200 if training exceeds 90 seconds.")

---
## 6.5 — Configure NHITS
**[Code With Me — 2 lines]**

> **Instructor note (3 min):** Students fill in `input_size` and `max_steps` from the loaded params dict. Walk through why `input_size = 2 * HORIZON` is the NeuralForecast convention — it gives the model two full horizon-lengths of lookback context. Then pause and check everyone is configured before running the training cell.

In [ ]:
# Configure NHITS — all params come from nhits_params loaded above
# input_size and max_steps are the two most important settings to understand

nhits_model = NHITS(
    h=nhits_params["h"],
    input_size=nhits_params["input_size"],        # __FILL_IN__: from nhits_params
    max_steps=nhits_params["max_steps"],          # __FILL_IN__: from nhits_params — primary runtime lever
    val_check_steps=nhits_params["val_check_steps"],
    early_stop_patience_steps=nhits_params["early_stop_patience_steps"],
    n_freq_downsample=nhits_params["n_freq_downsample"],
    learning_rate=nhits_params["learning_rate"],
    batch_size=nhits_params["batch_size"],
    random_seed=nhits_params["random_seed"],
    enable_progress_bar=nhits_params["enable_progress_bar"],
    enable_model_summary=nhits_params["enable_model_summary"],
)

nf = NeuralForecast(models=[nhits_model], freq="D")

print(f"NeuralForecast configured.")
print(f"  Model       : NHITS")
print(f"  h           : {nhits_params['h']}")
print(f"  input_size  : {nhits_params['input_size']}")
print(f"  max_steps   : {nhits_params['max_steps']}")

**Expected output:**
```
NeuralForecast configured.
  Model       : NHITS
  h           : 28
  input_size  : 56
  max_steps   : 300
```

---
## 6.6 — Run Cross-Validation on the Micro Subset
**[Watch Only]**

> **Instructor note (3 min):** Run this cell. **Watch the clock.** Target is < 60 seconds. Hard ceiling is 90 seconds. If it approaches 90, interrupt with `I, I` and jump to the Red Path in 6.7.
>
> While it trains: "NHITS is processing each CV window independently, training from scratch each time. In production you would train once and do rolling inference — not this. We are doing it this way because it matches the evaluation setup we used for every other model."

In [ ]:
%%time
# NHITS cross-validation on micro subset (50 series)
# Target runtime: < 60 seconds | Hard ceiling: 90 seconds
# If this cell exceeds 90 seconds: interrupt (I, I) and run the Red Path in 6.7
# NFPredictionIntervals requires refit=True — controlled via USE_INTERVALS in config.py

cv_dl_micro = nf.cross_validation(
    df=micro,
    n_windows=N_WINDOWS,
    step_size=STEP_SIZE,
    refit=True if USE_INTERVALS else REFIT,                                                                                         # __FILL_IN__: True when using NFPredictionIntervals, REFIT otherwise
    val_size=HORIZON,
    prediction_intervals=NFPredictionIntervals(n_windows=N_WINDOWS, method="conformal_distribution") if USE_INTERVALS else None,    # __FILL_IN__: Nixtla-native conformal intervals
    level=[80] if USE_INTERVALS else None,                                                                                          # __FILL_IN__: 80% prediction interval
)

print(f"NHITS CV complete. Shape: {cv_dl_micro.shape}")
print(f"Columns: {list(cv_dl_micro.columns)}")
print(cv_dl_micro.head(3).to_string())
GREEN_PATH_SUCCEEDED = True

**Expected output:**
```
NHITS CV complete. Shape: (4200, 7)
Columns: ['ds', 'cutoff', 'y', 'NHITS', 'NHITS-lo-80', 'NHITS-hi-80']
```

> **Instructor note:** NeuralForecast now returns interval columns directly — `NHITS-lo-80` and `NHITS-hi-80`. These are calibrated using held-out windows via `NFPredictionIntervals`, identical pattern to MLForecast in Module 5. The reshape in 6.8 just renames these columns to the schema format.

---
## 6.7 — Red Path Recovery
**[Watch Only]**

> **Instructor note:** If 6.6 succeeded, skip this cell. If it timed out or errored: uncomment and run. Explain that `dl_forecasts` below is populated either from the live run or the precomputed artifact — the rest of the notebook does not care which.

In [ ]:
# 🔴 RED PATH — run this cell only if 6.6 timed out or failed
# Loads the precomputed full-subset DL forecasts as a drop-in replacement

# from src.checkpointing import load_checkpoint
# cv_dl_micro = None  # Green Path result not available
# GREEN_PATH_SUCCEEDED = False
# print("Red Path active — micro NHITS results unavailable. Full-subset artifact loaded in 6.11.")

---
## 6.8 — Reshape NeuralForecast CV Output
**[Code With Me — 2 lines]**

> **Instructor note (2 min):** Same pattern as Module 5. NeuralForecast returns native interval columns `NHITS-lo-80` and `NHITS-hi-80` from `NFPredictionIntervals`. The reshape function just renames them to the schema format. Students fill in the two rename keys.
>
> Point out: this is structurally identical to `reshape_mlforecast_cv` — consistent schema across all model stages is what makes the leaderboard possible.

In [ ]:
def reshape_neuralforecast_cv(cv_df: pd.DataFrame, model_col: str, stage: str) -> pd.DataFrame:
    """
    Reshape NeuralForecast wide CV output to forecast schema.
    Interval columns (lo_80, hi_80) come from NFPredictionIntervals — Nixtla-native conformal calibration.
    Identical pattern to Module 5 reshape — same schema contract.
    """
    df = cv_df.reset_index().copy()
    df = df.rename(columns={
        model_col:              "y_hat",
        f"{model_col}-lo-80":  "lo_80",   # __FILL_IN__: rename native lo column to schema name
        f"{model_col}-hi-80":  "hi_80",   # __FILL_IN__: rename native hi column to schema name
    })
    df["model"] = "NHITS"
    df["stage"] = stage
    df["y_hat"] = df["y_hat"].clip(lower=0)
    df["lo_80"] = df["lo_80"].clip(lower=0)
    df["hi_80"] = df["hi_80"].clip(lower=0)

    return df[["unique_id", "ds", "y", "model", "y_hat", "lo_80", "hi_80", "cutoff", "stage"]]


if cv_dl_micro is not None:
    dl_micro = reshape_neuralforecast_cv(cv_dl_micro, model_col="NHITS", stage="dl")
    dl_micro_validated = validate_forecast(dl_micro, artifact_name="06_dl_micro")
    print(f"Reshaped DL forecasts: {dl_micro.shape}")
    print(f"Columns: {list(dl_micro.columns)}")
else:
    print("Green Path not available. Skipping micro reshape — using full artifact in 6.11.")
    dl_micro_validated = None

**Expected output (if Green Path succeeded):**
```
Reshaped DL forecasts: (4200, 9)
Columns: ['unique_id', 'ds', 'y', 'model', 'y_hat', 'lo_80', 'hi_80', 'cutoff', 'stage']
```

---
## 6.9 — Plot: NHITS vs LightGBM vs SeasonalNaive
**[Watch Only]**

> **Instructor note (2 min):** Plot all three on one chart. The visual comparison rarely gives a definitive answer — that is intentional. "Looking at a chart is not how we make model selection decisions. The leaderboard is."

In [ ]:
if dl_micro_validated is not None:
    sample_uid = top_series[0]
    sample_cut = dl_micro_validated["cutoff"].unique()[-1]

    actuals    = panel[panel["unique_id"] == sample_uid].set_index("ds")["y"]
    plot_start = sample_cut - pd.Timedelta(days=42)

    nhits_fcast = dl_micro_validated[
        (dl_micro_validated["unique_id"] == sample_uid) &
        (dl_micro_validated["cutoff"]    == sample_cut)
    ].set_index("ds")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(actuals[actuals.index >= plot_start].index,
            actuals[actuals.index >= plot_start].values,
            color="#333", linewidth=1.0, label="Actual", zorder=3)
    ax.plot(nhits_fcast.index, nhits_fcast["y_hat"],
            color="#F4511E", linewidth=1.5, linestyle="--", label="NHITS", zorder=4)
    ax.fill_between(nhits_fcast.index, nhits_fcast["lo_80"], nhits_fcast["hi_80"],
                    alpha=0.15, color="#F4511E", label="NHITS 80% interval")

    # Overlay LightGBM micro if available
    try:
        ml_micro_path = ARTIFACT_DIR / "05_ml_micro_forecasts.parquet"
        ml_micro = pd.read_parquet(ml_micro_path)
        lgbm_fcast = ml_micro[
            (ml_micro["unique_id"] == sample_uid) &
            (ml_micro["cutoff"]    == sample_cut)
        ].set_index("ds")
        if len(lgbm_fcast) > 0:
            ax.plot(lgbm_fcast.index, lgbm_fcast["y_hat"],
                    color="#7B1FA2", linewidth=1.2, linestyle=":", label="LightGBM", zorder=3)
    except Exception:
        pass

    ax.axvline(sample_cut, color="#999", linestyle=":", linewidth=1)
    ax.set_title(f"NHITS vs LightGBM — {sample_uid} (Window 3)", fontsize=11)
    ax.set_ylabel("Units sold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Green Path not available — plot skipped. Visual comparison available in Module 8.")

**Expected output (if Green Path succeeded):** NHITS forecast with shaded 80% interval, LightGBM dotted overlay, actuals in dark gray.

---
## 6.10 — Score the Micro DL Forecasts
**[Code With Me — 1 line]**

> **Instructor note (1 min):** Same scoring pattern as Module 5. Students fill in the `score_forecasts` call. If Red Path was taken, skip this cell gracefully — the full artifact scores in 6.11.

In [ ]:
if dl_micro_validated is not None:
    dl_micro_scores = score_forecasts(
        dl_micro_validated,
        subset_name=f"micro_{MICRO_SUBSET_N}",  # __FILL_IN__: call score_forecasts on dl_micro_validated
    )

    wmape_row  = dl_micro_scores[dl_micro_scores["metric"] == "wMAPE"].iloc[0]
    iscore_row = dl_micro_scores[dl_micro_scores["metric"] == "IntervalScore_80"].iloc[0]
    bias_row   = dl_micro_scores[dl_micro_scores["metric"] == "Bias"].iloc[0]

    print(f"NHITS on micro subset ({MICRO_SUBSET_N} series):")
    print(f"  wMAPE          : {wmape_row['score']:.4f}")
    print(f"  Interval Score : {iscore_row['score']:.4f}")
    print(f"  Bias           : {bias_row['score']:+.4f}  (+ = over-forecast, - = under-forecast)")
else:
    print("Micro scoring skipped — Red Path was taken. Full-subset scores in 6.11.")

**Expected output (if Green Path succeeded):**
```
NHITS on micro subset (50 series):
  wMAPE          : 0.XXXX
  Interval Score : X.XXXX
```

---
## 6.11 — Load Full-Subset DL Forecasts and Update Leaderboard
**[Watch Only]**

> **Instructor note (2 min):** Load the precomputed full-subset NHITS artifact and build the 6-model leaderboard. This is the first time all models appear on the same leaderboard. Take a moment here — Module 8 will revisit this table with the interval scores added.

In [ ]:
# 🔴 RED PATH (standard) — always load precomputed full-subset DL forecasts
dl_full = load_checkpoint("06_dl_forecasts")

dl_full_scores       = score_forecasts(dl_full,      subset_name=f"workshop_{WORKSHOP_SUBSET_N}")
baseline_full        = load_checkpoint("04_baseline_forecasts")
baseline_full_scores = score_forecasts(baseline_full, subset_name=f"workshop_{WORKSHOP_SUBSET_N}")
ml_full              = load_checkpoint("05_ml_forecasts")
ml_full_scores       = score_forecasts(ml_full,       subset_name=f"workshop_{WORKSHOP_SUBSET_N}")

leaderboard_6 = build_leaderboard([baseline_full_scores, ml_full_scores, dl_full_scores])

print("Six-model leaderboard (wMAPE):")
if "wMAPE" in leaderboard_6.columns:
    print(leaderboard_6[["model", "wMAPE"]].dropna().sort_values("wMAPE").to_string(index=False))
else:
    print(leaderboard_6.to_string(index=False))

**Expected output:**
```
Six-model leaderboard (wMAPE):
          model    wMAPE
       LightGBM   0.XXXX
          NHITS   0.XXXX
        AutoETS   0.XXXX
Chronos-t5-mini   0.XXXX
  SeasonalNaive   0.XXXX
          Naive   0.XXXX
```

> **Instructor note:** The relative ranking of NHITS and LightGBM is the most interesting result in the workshop. Both are global models trained on all series. The difference is architecture — and the gap is usually smaller than students expect.

---
## 6.12 — Save the Micro DL Artifact
**[Watch Only]**

> **Instructor note (30 sec):** Save if available. Quick.

In [ ]:
if dl_micro_validated is not None:
    micro_dl_path = ARTIFACT_DIR / "06_dl_micro_forecasts.parquet"
    dl_micro_validated.to_parquet(micro_dl_path, index=False)
    print(f"  ✓ Micro DL artifact saved : {micro_dl_path.name} ({len(dl_micro_validated):,} rows)")
print(f"  ✓ Full DL artifact loaded : 06_dl_forecasts.parquet ({len(dl_full):,} rows)")

---
## 6.13 — The Enterprise Cliff
**[Watch Only]**

> **Instructor note (2 min):** This is the last enterprise cliff before the final evaluation. The gap between NHITS and LightGBM on wMAPE directly motivates the "should we build this?" question in Module 8.

We trained NHITS in under 90 seconds on 50 series. In production, the cost profile is completely different.

**Training time scales with data, not series count:**  
NHITS processes every time step as a training sample. At 100,000 SKUs × 3 years of daily history, a single training run takes hours on CPU. This is a GPU job in production — and the infrastructure cost needs to be justified by the accuracy margin over LightGBM.

**Retraining cadence is a system design decision:**  
LightGBM can be retrained daily in minutes. NHITS cannot. In practice, neural forecasting models are retrained weekly or monthly and serve predictions from a frozen checkpoint in between. This introduces model staleness risk — especially around holidays or demand shocks.

**The wMAPE gap tells you whether to proceed:**  
If NHITS beats LightGBM by 2+ wMAPE points on your portfolio, the GPU compute cost is likely justified. If the gap is less than 1 point, LightGBM is the production choice — it is faster, cheaper, and more interpretable.

Look at your leaderboard. Make the call.

---

> **Instructor note — transition:** "Six models. Three stages. Now we judge uncertainty quality and make a final recommendation."
>
> Open `instructor_notebooks/07_prediction_intervals.ipynb`.